# Frozen A — attention geometry diagnostic (tối đa 200 ảnh TRAIN)
**Internet ON · GPU T4×2 · Save & Run All.** Hai GPU chia ảnh độc lập; không huấn luyện/DDP collectives.
Kiểm tra cụm danh từ trong **prompt legacy nguyên bản của A** có attention đúng vùng GT hay không.
Không thay prompt bằng UPT, không tạo caption mới, không cập nhật checkpoint.
Đầu vào: checkpoint A, legacy prompt cache, **JSON prompt gốc**, global visual cache, MSCOCO dataset JSON,
dataset noun-box TRAIN10k v1.1 đã dùng trong COCO routing B/R. Không cần ảnh gốc/UPT hidden/CHAIR.
Nếu cache không có văn bản prompt hoặc token-binding khác, notebook dừng và xuất review để kiểm tra.

In [ ]:
print('FROZEN ATTENTION GEOMETRY: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
NOUN_MANIFEST=Path('/kaggle/input/datasets/ducanh2403/coco-noun-box-train10k-v11')
# Cache .pt gốc chỉ có tokens/mask. Cần thêm JSON văn bản ĐÃ dùng để tạo cache đó.
# Đây là path được ghi trong notebook gốc; sửa nếu bạn mount JSON ở dataset khác.
PROMPT_TEXT=Path('/kaggle/input/models/ducanh2403/promt-cache/maxtext/default/1/prompt_cache.json')
OUTPUT=Path('/kaggle/working/attention_geometry_diagnostic_200')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,NOUN_MANIFEST/'manifest_receipt.json'):
    assert p.is_file(),f'Missing input: {p}; sửa đúng mount path trong cell này.'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
if not PROMPT_TEXT.is_file():
    sibling=LEGACY_PROMPT.parent/'prompt_cache.json'
    if sibling.is_file():PROMPT_TEXT=sibling
assert PROMPT_TEXT.is_file(),f'Thiếu JSON prompt GỐC: {PROMPT_TEXT}. Thêm model promt-cache hoặc JSON gốc; không dùng JSON UPT.'
assert not OUTPUT.exists(),'Giữ output cũ; dùng session/output mới.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-attention-geometry-'))/'repo'
LOG = Path('/tmp/upt_hidden_attention-geometry.log')
PROJECT_COMMIT = '973c22925826b50b1497a9502390dea96e2e621a'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'diagnostic_data.py': '15c0eab863ad6500436d25372e34e3395266774838db39b2bdb8e402ff150312', 'run_diagnostic.py': '382653c64ecc7047f908ae717d5d963efa177767ad8db4774093207e286ff731', 'test_diagnostic.py': '2604cef5b044b1b749f51960db1dc35238aef2a61c80c29227d50286e7a0a871', 'route_probe.py': '7cd9071bc4671873f027c66e4e243e9771947dba8b4ebe83dae4cc7d129783bc', 'audit.py': 'cadf9d172e92e1558020520399ec6013d1141b8ab82c34821d6ba4f0dfe244b4', 'manifest_policy.py': 'f29be2f60bedfff2da16b591d379fb065db8c5d87a23924cf432a31217ab88a6', 'manifest_policy_v11.py': '165106dc3d8e759d9515715cfa5754cd8a9cf70096384d84c1ac035849418547', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow'])
import torch
assert torch.cuda.device_count()==2,'Select GPU T4x2.'
run([sys.executable,'-u','test_diagnostic.py'],cwd=REPO)
# Synthetic attention observer comparison only. Actual A is checked without switching its MHA kernel.
run([sys.executable,'-u','-c',"from route_probe import numeric_checks; print(numeric_checks('cuda:0'))"],cwd=REPO)


In [ ]:
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),noun_manifest=str(NOUN_MANIFEST),prompt_text_json=str(PROMPT_TEXT),output=str(OUTPUT),
    implementation_sha256=SOURCE_HASHES,project_commit=PROJECT_COMMIT)
CONFIG=Path('/tmp/attention_geometry_config.json')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_diagnostic.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
        '--nproc_per_node=2','--max_restarts=0','run_diagnostic.py','worker','--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_diagnostic.py','finish','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        for name,source in SOURCES.items():
            target=OUTPUT/'source'/name;target.parent.mkdir(parents=True,exist_ok=True)
            target.write_text(source,encoding='utf-8')
        run([sys.executable,'-u','run_diagnostic.py','archive','--config',CONFIG],cwd=REPO)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)


Gửi `attention_geometry_diagnostic_200_review.zip` sau khi hoàn tất, kể cả khi dừng do thiếu metadata.
Chỉ `receipt.json` với DIAGNOSTIC_COMPLETE xác nhận chạy đủ; ZIP tồn tại chưa phải thành công.
`PREMISE_SIGNAL_GO_DESIGN_ONLY` chỉ cho phép bàn pilot tiếp, chưa chứng minh caption tăng hay phương pháp mới.
Thời gian chưa benchmark; không dùng con số 0 GPU. Model đọc cache CLIP, text encoder chỉ xác minh token-binding.
Nếu FAIL, gửi log; không đổi tolerance, bỏ bước kiểm hoặc tự bật thêm module/loss.